
纯torch代码实现vllm中默认参数下的Embedding全过程

In [1]:
import torch
import numpy as np
import json
import re

from examples.pytorch.context_parallel import vocab_size

ValueError: Error initializing torch.distributed using env:// rendezvous: environment variable RANK expected, but not set

In [ ]:
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"

texts = ["Hello Word, a test sentence"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Tokenizer

In [ ]:
# 读取Tokenizer配置
tokenize_config_file = "/Users/pengjunzhe/.cache/huggingface/hub/models--Qwen--Qwen3-Embedding-0.6B/snapshots/c54f2e6e80b2d7b7de06f51cec4959f6b3e03418/tokenizer_config.json"

with open(tokenize_config_file, encoding="utf-8") as f:
    tokenize_config = json.load(f)

In [ ]:
# 1.1 读取merges文件和vocab文件
# 加载tokenizer文件
merges_file = "/Users/pengjunzhe/.cache/huggingface/hub/models--Qwen--Qwen3-Embedding-0.6B/snapshots/c54f2e6e80b2d7b7de06f51cec4959f6b3e03418/merges.txt"
vocab_file = "/Users/pengjunzhe/.cache/huggingface/hub/models--Qwen--Qwen3-Embedding-0.6B/snapshots/c54f2e6e80b2d7b7de06f51cec4959f6b3e03418/vocab.json"

with open(vocab_file, "r", encoding="utf-8") as f:
    vocab = json.load(f)

with open(merges_file, "r", encoding="utf-8") as f:
    merges = [tuple(line.strip().split()) for line in f if line.strip() and not line.startswith("#")]



In [ ]:
len(vocab)

In [ ]:
# 1.2 创建 BPE rank dict
bpe_ranks = {pair: i for i, pair in enumerate(merges)}
bpe_ranks

In [ ]:
# 1.3 构建BPE分词函数
def get_pairs(word):
    pairs = set()
    prev_char = word[0]
    for char in word[1:]:
        pairs.add((prev_char, char))
        prev_char = char
    return pairs

def bpe(token):
    """对单个 token 应用 BPE"""
    word = tuple(token)
    pairs = get_pairs(word)

    while pairs:
        # 找到最小rank的pair
        min_pair = min(pairs, key=lambda pair: bpe_ranks.get(pair, float('inf')))
        if min_pair not in bpe_ranks:
            break
        first, second = min_pair
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word)-1 and word[i] == first and word[i+1] == second:
                new_word.append(first+second)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        word = tuple(new_word)
        if len(word) == 1:
            break
        pairs = get_pairs(word)
    return word

In [ ]:
# 构建tokenize函数
def tokenize(
        text,
        do_lower_case = False,
):
    # strip
    to_tokenize = str(text).strip()

    # Lowercase
    if do_lower_case:
        to_tokenize = to_tokenize.lower()

    print(f"Tokenizing: '{to_tokenize}'")

    """文本转 token ID 列表"""
    # 简单按空格拆分词，可以根据实际情况改进
    words = re.findall(r"\S+", to_tokenize)
    token_ids = []
    for word in words:
        for tok in bpe(word):
            if tok in vocab:
                token_ids.append(vocab[tok])
            else:
                # 未知 token 可用 <unk> 或 0
                token_ids.append(vocab.get("<unk>", 0))
    return token_ids

tokenize("Hello Word, a test sentence")

In [ ]:
# padding_strategy = latest
# truncation_strategy = 'longest_first'
# max_length = 32768
# kwargs = {}
# add_special_tokens = True
# return_tensors = pt
[  9707,   9322,     11,    264,   1273,  11652, 151643]

In [ ]:
# 加载model
model = AutoModel.from_pretrained(MODEL_NAME)

In [ ]:
model.to(device)

In [ ]:
with torch.no_grad():
    outputs = model(**features)
    print(outputs)

In [ ]:
print(outputs.hidden_states[-1])

In [ ]:
# sentense_transformers的池化部分
token_embeddings = outputs.hidden_states[-1]
print(token_embeddings)
attention_mask = (
            features["attention_mask"]
            if "attention_mask" in features
            else torch.ones(token_embeddings.shape[:-1], device=token_embeddings.device, dtype=torch.int64)
        )
print(attention_mask)

In [ ]:
output_vectors = []
input_mask_expanded = (
    attention_mask.unsqueeze(-1).expand(token_embeddings.size()).to(token_embeddings.dtype)
)
print(input_mask_expanded)
sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
print(sum_embeddings)

In [ ]:
# If tokens are weighted (by WordWeights layer), feature 'token_weights_sum' will be present
if "token_weights_sum" in features:
    sum_mask = features["token_weights_sum"].unsqueeze(-1).expand(sum_embeddings.size())
else:
    sum_mask = input_mask_expanded.sum(1)

sum_mask = torch.clamp(sum_mask, min=1e-9)
print(sum_mask)

In [ ]:
# if self.pooling_mode_mean_tokens:
output_vectors.append(sum_embeddings / sum_mask)
# if self.pooling_mode_mean_sqrt_len_tokens:
# output_vectors.append(sum_embeddings / torch.sqrt(sum_mask))
print(output_vectors)

In [ ]:
output_vector = torch.cat(output_vectors, 1)
features["sentence_embedding"] = output_vector

In [ ]:
features

In [ ]:
# Sentence embeddings
embeddings = features["sentence_embedding"]
print(embeddings)
embeddings = embeddings.detach()
print(embeddings)
# if normalize_embeddings:
#     embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

In [ ]:
# if convert_to_numpy:
embeddings = embeddings.cpu()
print(embeddings)
all_embeddings = embeddings

In [ ]:
# elif convert_to_numpy:
# if not isinstance(all_embeddings, np.ndarray):
# if all_embeddings and all_embeddings[0].dtype == torch.bfloat16:
# all_embeddings = np.asarray([emb.float().numpy() for emb in all_embeddings])
# else:
import numpy as np
all_embeddings = np.asarray([emb.numpy() for emb in all_embeddings])
print(all_embeddings)

In [ ]:
result_embedding = all_embeddings[0]
print(result_embedding)

In [ ]:
result_embedding.shape == (768,)

In [ ]:
abs(np.sum(result_embedding) - 7.9811716) < 0.002